# Attention visualization
Encoder self-attention, decoder self-attention, and encoder-decoder (cross) attention for one sentence. Run the sample cell again to get the next validation sentence, or pass your own text.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
PROJECT = '/content/drive/MyDrive/en_ar_transformer'
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
!pip -q install datasets tokenizers tensorboard sacrebleu altair

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 8.4 MB/s eta 0:00:00


In [4]:
import torch, pandas as pd, altair as alt, warnings
from config import get_config
from train import load_data, greedy_decode
from decoding import Translator
warnings.filterwarnings("ignore")
alt.data_transformers.disable_max_rows()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
config = get_config()
translator = Translator(config, device)
model, tok_src, tok_tgt = translator.model, translator.tok_src, translator.tok_tgt
_, val_dl, _, _, _ = load_data(config)
val_iter = iter(val_dl)

Loaded /content/drive/MyDrive/en_ar_transformer/weights_v2/latest.pt  (trained epochs: 12)


README.md:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

ar-en/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  214kB            

ar-en/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

ar-en/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 99.3MB            

...

Generating train split:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/300000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/300000 [00:00<?, ? examples/s]

Kept 261190/261193 pairs after token-length filter
Kept 29022/29022 pairs after token-length filter


In [5]:
def load_sample(text=None):
    """Greedy-decode one sentence so the attention maps of that run are stored in the model."""
    model.eval()
    if text is None:
        batch = next(val_iter)
        src = batch["encoder_input"].to(device)
        mask = batch["encoder_attention_mask"].to(device)
        print("Source:", batch["src_text"][0])
        print("Target:", batch["tgt_text"][0])
    else:
        src, mask = translator.encode_source(text)
        print("Source:", text)

    with torch.no_grad():
        out = greedy_decode(model, src, mask, tok_src, tok_tgt, config["seq_len"], device)
    print("Model :", tok_tgt.decode(out.tolist(), skip_special_tokens=True))

    enc_tokens = [tok_src.id_to_token(i) for i in src[0].tolist()]
    dec_tokens = [tok_tgt.id_to_token(i) for i in out.tolist()]
    enc_len = int(mask.sum())                       # non-pad source tokens
    return enc_tokens, dec_tokens, enc_len

In [6]:
def get_attn_map(attn_type, layer, head):
    if attn_type == "encoder":
        attn = model.encoder.layers[layer].self_attention.attention_scores
    elif attn_type == "decoder":
        attn = model.decoder.layers[layer].self_attention.attention_scores
    else:  # "cross"
        attn = model.decoder.layers[layer].cross_attention.attention_scores
    return attn[0, head].detach().cpu()


def mtx2df(m, row_tokens, col_tokens, max_row, max_col):
    rows = []
    for r in range(min(m.shape[0], max_row)):
        for c in range(min(m.shape[1], max_col)):
            rt = row_tokens[r] if r < len(row_tokens) else "<blank>"
            ct = col_tokens[c] if c < len(col_tokens) else "<blank>"
            rows.append((r, c, float(m[r, c]), "%.3d %s" % (r, rt), "%.3d %s" % (c, ct)))
    return pd.DataFrame(rows, columns=["row", "column", "value", "row_token", "col_token"])


def attn_chart(attn_type, layer, head, row_tokens, col_tokens, max_row, max_col, size=260):
    df = mtx2df(get_attn_map(attn_type, layer, head), row_tokens, col_tokens, max_row, max_col)
    return (alt.Chart(df).mark_rect()
            .encode(x=alt.X("col_token", axis=alt.Axis(title="")),
                    y=alt.Y("row_token", axis=alt.Axis(title="")),
                    color="value",
                    tooltip=["row", "column", "value", "row_token", "col_token"])
            .properties(height=size, width=size, title=f"Layer {layer} Head {head}")
            .interactive())


def all_maps(attn_type, layers, heads, row_tokens, col_tokens, max_row, max_col):
    return alt.vconcat(*[
        alt.hconcat(*[attn_chart(attn_type, l, h, row_tokens, col_tokens, max_row, max_col) for h in heads])
        for l in layers
    ])

In [7]:
enc_tokens, dec_tokens, enc_len = load_sample()          # or load_sample("I will see you tomorrow.")
dec_len = get_attn_map("decoder", 0, 0).shape[0]          # decoder positions that were actually computed
N = 20                                                    # show at most N tokens per axis
layers = [0, 1, 2]                                        # model has 5 layers: 0..4
heads  = list(range(config["num_heads"]))                 # 8 heads

Source: Haven't you ever heard that the unexamined life is not worth living?
Target: ما أنت أبدا سمعت ذلك الحياة الغير مفحوصة ألن يساوي يعيش؟
Model : ألم تسمع ذلك من قبل حياة لا يستحق الحياة التي لا يستحق أن تكون على الإطلاق ؟


### Encoder self-attention (source tokens attend to source tokens)

In [8]:
all_maps("encoder", layers, heads, enc_tokens, enc_tokens, min(N, enc_len), min(N, enc_len))

alt.VConcatChart(...)

### Decoder self-attention (generated Arabic tokens, causal)

In [9]:
all_maps("decoder", layers, heads, dec_tokens, dec_tokens, min(N, dec_len), min(N, dec_len))

alt.VConcatChart(...)

### Encoder-decoder (cross) attention
Rows = generated Arabic tokens, columns = English source tokens. This is the one that shows alignment.

In [10]:
all_maps("cross", layers, heads, dec_tokens, enc_tokens, min(N, dec_len), min(N, enc_len))

alt.VConcatChart(...)